In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from ising.blume_capel import BlumeCapel
from ising.model import Ising

from climate_attitudes import configure_mpl
from climate_attitudes.dataset import Dataset
from climate_attitudes.settings import Config

FONT_PATH = Path("../fonts")
configure_mpl(FONT_PATH)

In [ ]:
config = Config(_env_file="../.env")
dataset = Dataset.load(config, name="reduced_no_imputation", with_imputation=False)
resp = dataset.response.collect()

In [ ]:
df = resp.select(
    (pl.col("cc_pol_tax", "cc_pol_car", "cvcc6", "cvcc9_cc") - 3).clip(-1, 1),
    # pl.col("cc_ica").replace({0: -1, 2: 1}),
    # pl.col("pol7").replace({1: -1, 2: 1}),
)

### Checking for interaction effects

Compare expected probability of total support/opposition with the observed proportion.

In [ ]:
data = df.to_numpy()

# Calculate P(X_i = 1)
p = ((data + 1) / 2).mean(axis=0)

# Probability of observing all 1
p_all_1 = np.prod(p)

# Observed proportion of all 1
p_all_1_obs = len(data[data.sum(axis=1) == 6]) / len(data)

print(
    f"Expected probability of total support under independence "
    f"assumption: {p_all_1:.4f}"
)
print(f"Observed probability of total support: {p_all_1_obs:.4f}")

# Probability of observing all 0
p_all_0 = np.prod(1 - p)

# Observed proportion of all 0
p_all_0_obs = len(data[data.sum(axis=1) == 0]) / len(data)

print(
    f"Expected probability of total opposition under independence "
    f"assumption: {p_all_0:.4f}"
)
print(f"Observed probability of total opposition: {p_all_0_obs:.4f}")

### Fitting symmetric Ising model

We indeed see strong interactions between most pairs of variables in the fit Ising model. Notable exceptions are:

- `cc_pol_tax` with `cc_ica`,
- `cc_pol_car` with `cc_ica` and `cvcc9_cc`, and
- `cvcc9_cc` with `pol7`.

After adjusting for pairwise interactions, `pol7` and `cc_pol_tax` both exhibit strong negative local field terms, even though for both variables 'support' is the predominant response. This may indicate that in-general participants tend to oppose environmental regulation (`pol7`) and emissions taxes (`cc_pol_tax`), but concurrent support for other policies confers _recorded_ support for these.

**Note:** These results are not stable when we look at only W3 or only W4.

In [ ]:
ising_data = df.filter(pl.all_horizontal(pl.all() != 0)).to_numpy()
model = Ising.fit(ising_data)
fig = model.plot(df.columns, figsize=(5, 3.5))
plt.show()

### Symmetric Blume-Capel model

In [ ]:
bc_data = df.to_numpy()
model = BlumeCapel.fit(ising_data)
fig = model.plot(df.columns, figsize=(5, 3.5))
plt.show()